# WASB PREENTRENADO sobre nuestro clip (sin fine-tune)

El paso decisivo y barato: medir si el detector TEMPORAL preentrenado (wasb_soccer)
ya supera el techo de YOLO (56% de frames con la pelota, 45% post-Viterbi) en
NUESTRO footage, ANTES de invertir en fine-tune.

⚠️ El repo WASB tiene el entrenamiento incompleto (`train_and_test.py` con
`assert 0`), pero la EVAL/inferencia anda. Esto usa solo eso.

**Prerrequisito:** `spain-france-test3min.mp4` (el clip) en
`MyDrive/football_analytics/videos/`. (Es el clip de 3 min, no el partido completo.)

## 1) Setup: repos + pesos + deps

In [ ]:
import os
# nuestro repo (para make_wasb_dataset.py + los labels)
if not os.path.exists('/content/ncf_event_tracker'):
    !git clone -q --branch events-model https://github.com/pipachiesa/ncf_event_tracker.git /content/ncf_event_tracker
!cd /content/ncf_event_tracker && git pull -q origin events-model
# repo WASB
if not os.path.exists('/content/WASB-SBDT'):
    !git clone -q https://github.com/nttcom/WASB-SBDT.git /content/WASB-SBDT
!pip install -q hydra-core omegaconf gdown
# pesos preentrenados soccer (Google Drive)
os.makedirs('/content/pretrained_weights', exist_ok=True)
W='/content/pretrained_weights/wasb_soccer_best.pth.tar'
if not os.path.exists(W):
    !gdown -q 1pg0MpMtKZ6ziYEr4oyfKYPOO3hjLw94l -O {W}
from google.colab import drive; drive.mount('/content/drive')
print('OK, peso:', os.path.exists(W), os.path.getsize(W) if os.path.exists(W) else 0, 'bytes')

## 2) Armar nuestro clip en formato soccer (frames 0-based + labels como annos)
Usa el `make_wasb_dataset.py` ya arreglado (0-based, estructura plana).

In [ ]:
VIDEO='/content/drive/MyDrive/football_analytics/videos/spain-france-test3min.mp4'
assert os.path.exists(VIDEO), f'FALTA el clip: {VIDEO}'
ROOT='/content/wasb_data/soccer'; CLIP='clip3min'
!cd /content/ncf_event_tracker && python3 events_model/make_wasb_dataset.py \
    --video "{VIDEO}" \
    --labels events_model/dataset/ball_gt/spain-france_ball_labels.csv \
    --out {ROOT} --clip {CLIP} --stride 2
print('frames:', len(os.listdir(f'{ROOT}/frames/{CLIP}')), '  anno:', os.path.exists(f'{ROOT}/annos/{CLIP}.xml'))

## 3) Patch a eval.py: volcar las predicciones a CSV
(Inserta un dump de `result_dict` -> `/content/wasb_preds.csv` sin tocar la lógica.)

In [ ]:
ev='/content/WASB-SBDT/src/runners/eval.py'
src=open(ev).read()
anchor="log.info('Time:{:.1f}(sec)'.format(t_elapsed))"
dump = anchor + '''
    import csv as _csv, os as _os
    with open('/content/wasb_preds.csv','w',newline='') as _fh:
        _w=_csv.writer(_fh); _w.writerow(['frame_file','x','y','score','visi'])
        for _p in sorted(result_dict.keys()):
            _r=result_dict[_p]
            _w.writerow([_os.path.basename(_p), _r['x'], _r['y'], _r['score'], int(_r['visi'])])
    log.info('DUMPED preds -> /content/wasb_preds.csv ({} frames)'.format(len(result_dict)))'''
if 'wasb_preds.csv' not in src:
    assert anchor in src, 'no encontre el anchor; pegame las lineas alrededor de result_dict en eval.py'
    open(ev,'w').write(src.replace(anchor, dump, 1)); print('patch aplicado')
else:
    print('ya estaba parcheado')

## 4) Correr WASB preentrenado (step=1 = predicción densa por frame)

In [ ]:
# apunta el dataset soccer a NUESTRO clip (train y test al mismo, eval solo usa test)
!cd /content/WASB-SBDT/src && python3 main.py --config-name=eval \
    dataset=soccer model=wasb \
    detector.model_path=/content/pretrained_weights/wasb_soccer_best.pth.tar \
    detector.step=1 \
    'runner.gpus=[0]' \
    dataset.root_dir={ROOT} \
    'dataset.train.videos=[{CLIP}]' 'dataset.test.videos=[{CLIP}]' 2>&1 | tail -40

## 5) Bajar las predicciones

In [ ]:
from google.colab import files
import os
if os.path.exists('/content/wasb_preds.csv'):
    print('lineas:', sum(1 for _ in open('/content/wasb_preds.csv')))
    files.download('/content/wasb_preds.csv')
else:
    print('NO se genero wasb_preds.csv — pegame el output de la celda 4')

## 6) Qué me pasás
- `wasb_preds.csv` (frame_file, x, y, score, visi).
- El output de la celda 4 (trae también la métrica nativa de WASB contra nuestros labels).

Yo lo convierto a candidatos, corro `eval_ball.py` (misma regla que YOLO) y lo
fusiono con el Viterbi. Comparamos contra el 56%/45% de YOLO.

⚠️ Si algo falla (deps, config, el patch), pegame el error y lo arreglo — no
pude probar esto local (no hay GPU/WASB acá).